# McCain Cold Chain — 00 Setup

**Business context:** McCain Foods ships ~1M frozen-food pallets/year from Florenceville-Bristol, NB to 40+ Canadian retailer DCs. VP Supply Chain Quality **Karen Macmillan** owns an annual $12M exposure to temperature excursions — product claims, retailer chargebacks, and brand erosion.

**Before state (current):** excursion detection is manual — a carrier emails reefer logs days after delivery, a quality analyst cross-references SAP shipments in Excel, and root cause takes 3–5 business days. ~18% of excursions are caught only when a retailer complaint arrives.

**After state (this demo):** real-time IoT telemetry from Lakebase → Lakeflow pipeline → ML early-warning → AI root-cause analysis → ops dashboard. Goal: **reduce excursion exposure by 40% ($4.8M) and cut root-cause time from 5 days to 30 minutes.**

This notebook provisions:
1. **Lakebase (Postgres) instance** for raw IoT telemetry — the OLTP system that trucks phone home to.
2. **Unity Catalog schema** for analytical tables (shipments, products, complaints).
3. **Seed data**: ~150K reefer telemetry rows + 120 shipments + 50 SKUs + 28 complaint emails, with 3 pre-seeded excursion scenarios (door stuck open, reefer unit failure, heat spike).

Run this once, then notebooks 01 → 06 in order.

In [0]:
%pip install -q "psycopg[binary]" databricks-sdk --upgrade
dbutils.library.restartPython()

In [0]:
LAKEBASE_INSTANCE_NAME = "mccain-cold-chain-sju"
LAKEBASE_CAPACITY = "CU_1"          # smallest tier; ~$0.40/hr
UC_CATALOG = "serverless_stable_qr9if1_catalog"
UC_SCHEMA = "mccain_cold_chain_sju"
PG_DB = "databricks_postgres"
PG_SCHEMA = "iot"
TELEMETRY_STEP_MIN = 1              # 1-minute granularity → ~150K rows

# Store config as widgets so downstream notebooks can inherit
dbutils.widgets.text("uc_catalog", UC_CATALOG)
dbutils.widgets.text("uc_schema", UC_SCHEMA)
dbutils.widgets.text("lakebase_instance", LAKEBASE_INSTANCE_NAME)

## 1. Create (or reuse) the Lakebase instance

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance

w = WorkspaceClient()
me = w.current_user.me().user_name
print(f"Current user: {me}")

try:
    inst = w.database.get_database_instance(name=LAKEBASE_INSTANCE_NAME)
    print(f"Reusing existing instance: {inst.name} (state={inst.state})")
except Exception:
    print(f"Creating new Lakebase instance '{LAKEBASE_INSTANCE_NAME}' ({LAKEBASE_CAPACITY})…")
    inst = w.database.create_database_instance(
        database_instance=DatabaseInstance(
            name=LAKEBASE_INSTANCE_NAME,
            capacity=LAKEBASE_CAPACITY,
        )
    )

def _state_name(s) -> str:
    return s.name if hasattr(s, "name") else str(s).split(".")[-1]

deadline = time.time() + 900
while _state_name(inst.state) != "AVAILABLE":
    if time.time() > deadline:
        raise TimeoutError(f"Instance did not become AVAILABLE in 15 min — current state: {inst.state}")
    print(f"  state={_state_name(inst.state)} — waiting…")
    time.sleep(20)
    inst = w.database.get_database_instance(name=LAKEBASE_INSTANCE_NAME)

print(f"\n✅ Lakebase READY. Endpoint: {inst.read_write_dns}")

## 2. Open a Postgres connection using the current user's OAuth token

In [0]:
import uuid
import psycopg

cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()),
    instance_names=[LAKEBASE_INSTANCE_NAME],
)

def pg_connect(dbname: str = PG_DB, autocommit: bool = True) -> psycopg.Connection:
    return psycopg.connect(
        host=inst.read_write_dns,
        dbname=dbname,
        user=me,
        password=cred.token,
        sslmode="require",
        autocommit=autocommit,
    )

with pg_connect() as c:
    with c.cursor() as cur:
        cur.execute("select version();")
        print(f"✅ Connected: {cur.fetchone()[0]}")

## 3. Create the IoT schema + tables in Lakebase

In [0]:
DDL = f"""
CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA};

DROP TABLE IF EXISTS {PG_SCHEMA}.reefer_telemetry;
DROP TABLE IF EXISTS {PG_SCHEMA}.door_events;
DROP TABLE IF EXISTS {PG_SCHEMA}.gps_pings;
DROP TABLE IF EXISTS {PG_SCHEMA}.trailers;

CREATE TABLE {PG_SCHEMA}.trailers (
    trailer_id       text PRIMARY KEY,
    carrier_id       text NOT NULL,
    reefer_model     text NOT NULL,
    capacity_pallets int  NOT NULL,
    install_date     date NOT NULL
);

CREATE TABLE {PG_SCHEMA}.reefer_telemetry (
    trailer_id      text        NOT NULL,
    shipment_id     text        NOT NULL,
    ts              timestamptz NOT NULL,
    set_point_c     numeric(5,2),
    supply_air_c    numeric(5,2),
    return_air_c    numeric(5,2),
    compressor_on   boolean,
    fuel_level_pct  numeric(5,2),
    alarm_code      text
);
CREATE INDEX ON {PG_SCHEMA}.reefer_telemetry (trailer_id, ts);
CREATE INDEX ON {PG_SCHEMA}.reefer_telemetry (shipment_id);

CREATE TABLE {PG_SCHEMA}.door_events (
    event_id     text PRIMARY KEY,
    trailer_id   text NOT NULL,
    shipment_id  text NOT NULL,
    ts           timestamptz NOT NULL,
    event        text NOT NULL,
    location     text
);

CREATE TABLE {PG_SCHEMA}.gps_pings (
    trailer_id   text NOT NULL,
    shipment_id  text NOT NULL,
    ts           timestamptz NOT NULL,
    lat          numeric(9,5),
    lon          numeric(9,5),
    speed_kmh    numeric(5,1)
);
CREATE INDEX ON {PG_SCHEMA}.gps_pings (shipment_id, ts);
"""

with pg_connect() as c:
    with c.cursor() as cur:
        cur.execute(DDL)
print("✅ IoT schema + tables created in Lakebase.")

## 4. Generate synthetic data

Uses the `generate` module sitting next to this notebook. Deterministic (seed=42) so the same excursion scenarios reproduce every run.

In [0]:
import os, sys
notebook_dir = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
sys.path.insert(0, f"/Workspace{notebook_dir}/data")
import importlib, generate
importlib.reload(generate)

tables = generate.build_all(out_dir=None, telemetry_step_min=TELEMETRY_STEP_MIN)

# --- EXECUTION EVIDENCE: print row counts for every generated table ---
print("\n" + "="*60)
print("GENERATED DATA SUMMARY")
print("="*60)
total_rows = 0
for name, df in tables.items():
    rows = len(df)
    total_rows += rows
    print(f"  {name:<22} {rows:>8,} rows")
print(f"  {'TOTAL':<22} {total_rows:>8,} rows")
print("="*60)

## 5. Bulk-load IoT data into Lakebase via COPY

In [0]:
import io

def copy_df(conn, df, qualified_table: str, cols: list[str]):
    buf = io.StringIO()
    df[cols].to_csv(buf, index=False, header=False, na_rep="\\N")
    buf.seek(0)
    with conn.cursor() as cur:
        with cur.copy(
            f"COPY {qualified_table} ({', '.join(cols)}) FROM STDIN WITH (FORMAT csv, NULL '\\N')"
        ) as cpy:
            cpy.write(buf.read())

with pg_connect(autocommit=False) as conn:
    copy_df(conn, tables["trailers"], f"{PG_SCHEMA}.trailers",
            ["trailer_id", "carrier_id", "reefer_model", "capacity_pallets", "install_date"])
    copy_df(conn, tables["reefer_telemetry"], f"{PG_SCHEMA}.reefer_telemetry",
            ["trailer_id", "shipment_id", "ts", "set_point_c", "supply_air_c", "return_air_c",
             "compressor_on", "fuel_level_pct", "alarm_code"])
    copy_df(conn, tables["door_events"], f"{PG_SCHEMA}.door_events",
            ["event_id", "trailer_id", "shipment_id", "ts", "event", "location"])
    copy_df(conn, tables["gps_pings"], f"{PG_SCHEMA}.gps_pings",
            ["trailer_id", "shipment_id", "ts", "lat", "lon", "speed_kmh"])
    conn.commit()

# --- EXECUTION EVIDENCE: verify Lakebase row counts ---
print("\n" + "="*60)
print("LAKEBASE LOAD VERIFICATION")
print("="*60)
with pg_connect() as c:
    with c.cursor() as cur:
        for t in ["trailers", "reefer_telemetry", "door_events", "gps_pings"]:
            cur.execute(f"SELECT count(*) FROM {PG_SCHEMA}.{t}")
            print(f"  {PG_SCHEMA}.{t:<18} = {cur.fetchone()[0]:>8,} rows")
print("✅ All IoT data loaded to Lakebase")

## 6. Create the UC schema + land analytical tables

In [0]:
# Catalog already exists — just ensure schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")
spark.sql(f"USE {UC_CATALOG}.{UC_SCHEMA}")

# --- EXECUTION EVIDENCE: show row counts for each analytical table ---
print("\n" + "="*60)
print("UNITY CATALOG ANALYTICAL TABLES")
print("="*60)
for name in ["carriers", "retailer_dcs", "product_master", "shipments", "customer_complaints"]:
    sdf = spark.createDataFrame(tables[name])
    (sdf.write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.{name}"))
    count = sdf.count()
    print(f"  {UC_CATALOG}.{UC_SCHEMA}.{name:<20} = {count:>6} rows")
print("✅ All analytical tables written to Unity Catalog")

## Setup Complete ✅

| What | Where |
|---|---|
| Raw IoT (OLTP) | Lakebase `mccain-cold-chain-sju` → schema `iot` |
| Analytical | `main.mccain_cold_chain_sju` |

**Execution evidence produced above**: row counts for every generated table, Lakebase load verification, and UC table write confirmation.

**Next: 01_lakebase_to_lakehouse** — read IoT from Lakebase and land bronze Delta.